# Data download - firme din Romania

Notebook pentru testarea si implementarea descarcarii de date despre firme din diverse surse (vezi `requirements.txt`).

Fiecare sursa noua de date primeste propriile celule (o celula markdown de titlu + una sau mai multe celule de cod).

Toate sursele sunt controlate din celula de configurare de mai jos (`ACTIVE_SOURCES`), astfel incat sa putem tine tot codul intr-un singur notebook, dar sa rulam efectiv doar sursele de interes la un moment dat.

In [ ]:
import IPython
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 1100)
pd.set_option('display.width', 1000)
# Disable the SettingWithCopyWarning
pd.options.mode.chained_assignment = None

from itables import init_notebook_mode
import itables.options

itables.options.lengthMenu = [5, 10, 15, 25, 50]
init_notebook_mode(all_interactive=True)

import numpy as np
import matplotlib.pyplot as plt

from IPython.display import HTML, display

# Tema inchisa pentru tabelele itables: fundal negru, text alb, headere/sageti de sortare galbene.
# Sortarea pe coloane ramane activa implicit (DataTables) - click pe header pentru asc/desc.
# Acopera atat clasele DataTables 1.x (dataTables_*) cat si cele 2.x (dt-*), pt compatibilitate.
display(HTML("""
<style>
table.dataTable, table.dataTable thead, table.dataTable tbody,
table.dataTable thead th, table.dataTable thead td,
table.dataTable tbody th, table.dataTable tbody td {
    background-color: #000000 !important;
    color: #ffffff !important;
    border-color: #444444 !important;
}

table.dataTable thead th, table.dataTable thead td {
    color: #ffd700 !important;
}

table.dataTable tbody tr:hover td {
    background-color: #222222 !important;
}

.dataTables_wrapper, .dt-container,
.dataTables_wrapper .dataTables_info, .dt-container .dt-info,
.dataTables_wrapper .dataTables_length, .dt-container .dt-length,
.dataTables_wrapper .dataTables_filter, .dt-container .dt-search,
.dataTables_wrapper .dataTables_paginate, .dt-container .dt-paging {
    color: #ffffff !important;
}

.dataTables_wrapper .dataTables_filter input, .dt-container .dt-search input,
.dataTables_wrapper .dataTables_length select, .dt-container .dt-length select,
.dt-input {
    background-color: #000000 !important;
    color: #ffffff !important;
    border: 1px solid #666666 !important;
}

.dataTables_wrapper .dataTables_paginate .paginate_button,
.dt-container .dt-paging .dt-paging-button {
    color: #ffffff !important;
    background: transparent !important;
    border-color: #444444 !important;
}

.dataTables_wrapper .dataTables_paginate .paginate_button.current,
.dt-container .dt-paging .dt-paging-button.current {
    color: #000000 !important;
    background: #ffd700 !important;
    border-color: #ffd700 !important;
}

.dataTables_wrapper .dataTables_paginate .paginate_button.disabled,
.dt-container .dt-paging .dt-paging-button.disabled {
    color: #666666 !important;
}
</style>
"""))

In [ ]:
# Insert 30 seconds delay to allow the DataTables to render properly in Jupyter Notebook
import time
time.sleep(30)

In [ ]:
import requests

## Configurare surse active

Seteaza pe `True` sursele pe care vrei sa le rulezi acum. Poti activa una singura sau mai multe simultan.

In [ ]:
ACTIVE_SOURCES = {
    "onrc": False,          # onrc.ro
    "mfinante": False,      # mfinante.ro
    "portal_just": False,   # portal.just.ro
    "listafirme": False,    # listafirme.ro
    "totalfirme": False,    # totalfirme.ro
    "data_gov_ro": True,    # data.gov.ro (Guvernul Romaniei) - situatii financiare 2025
    "firme_data_gov_ro": True,  # data.gov.ro (Guvernul Romaniei) - firme inregistrate la Registrul Comertului
}


def is_active(source: str) -> bool:
    return ACTIVE_SOURCES.get(source, False)


print("Surse active:", [s for s, v in ACTIVE_SOURCES.items() if v])

## Sursa: ONRC (onrc.ro)

In [ ]:
if is_active("onrc"):
    # TODO: implementare descarcare date de pe onrc.ro
    pass
else:
    print("onrc: sarit (dezactivat in ACTIVE_SOURCES)")

## Sursa: Ministerul Finantelor (mfinante.ro)

In [ ]:
if is_active("mfinante"):
    # TODO: implementare descarcare date de pe mfinante.ro
    pass
else:
    print("mfinante: sarit (dezactivat in ACTIVE_SOURCES)")

## Sursa: Portal Just (portal.just.ro)

In [ ]:
if is_active("portal_just"):
    # TODO: implementare descarcare date de pe portal.just.ro
    pass
else:
    print("portal_just: sarit (dezactivat in ACTIVE_SOURCES)")

## Sursa: ListaFirme (listafirme.ro)

In [ ]:
if is_active("listafirme"):
    # TODO: implementare descarcare date de pe listafirme.ro
    pass
else:
    print("listafirme: sarit (dezactivat in ACTIVE_SOURCES)")

## Sursa: TotalFirme (totalfirme.ro)

In [ ]:
if is_active("totalfirme"):
    # TODO: implementare descarcare date de pe totalfirme.ro
    pass
else:
    print("totalfirme: sarit (dezactivat in ACTIVE_SOURCES)")

## Sursa: Guvernul Romaniei - Situatii financiare (data.gov.ro)

Dataset (2025): https://data.gov.ro/ro/dataset/situatii_financiare_2025

Fiecare an are propriul dataset pe data.gov.ro (URL-uri si ID-uri de resurse diferite), asa ca fisierele sunt organizate pe subfoldere per an: `data.gov.ro/Finance_<an>/` (ex. `Finance_2025/`, `Finance_2024/`, ...).

Fiecare set de date vine intr-o pereche de fisiere:
- fisierul `.txt` (delimitat prin virgula) contine efectiv datele, cu coloane codificate (`CUI`, `CAEN`, `I1`, `I2`, ...);
- fisierul `.csv` (delimitat prin `;`) contine descrierea coloanelor (`descriere;cod`), fara header.

`ANI_ACTIVI` (mai jos) controleaza ce ani se incarca efectiv; `FINANCIARE_FILES` are cate o intrare per an. Pentru a adauga un an nou (2024, 2023, 2026, ...): descarca fisierele acelui an de pe data.gov.ro, adauga anul in `ANI_ACTIVI` si completeaza `FINANCIARE_FILES[an]` cu fisierele + URL-urile lui.

Implicit (`DOWNLOAD_MISSING_FILES = False`), celula de mai jos doar incarca fisierele deja existente in `data.gov.ro/Finance_<an>/` — nu face niciun apel de retea. Seteaza `DOWNLOAD_MISSING_FILES = True` daca vrei sa descarci automat fisierele care lipsesc.

In [ ]:
import re
import unicodedata
from pathlib import Path

DATA_GOV_RO_BASE_DIR = Path("/Users/tudor/Documents/Data-for-Projects/Cercetare-Research/data.gov.ro")

# Daca False (implicit), nu se face niciun apel de retea: se incarca doar fisierele deja
# prezente in data.gov.ro/Finance_<an>/. Seteaza pe True doar cand vrei sa (re)descarci fisierele lipsa.
DOWNLOAD_MISSING_FILES = False

# Anii pentru care incarcam efectiv datele. Adauga un an nou aici doar dupa ce completezi
# FINANCIARE_FILES[an] mai jos cu fisierele si URL-urile lui (fiecare an e un dataset separat pe data.gov.ro).
ANI_ACTIVI = [2025]

# An -> {cheie: ((fisier date .txt, url date), (fisier descriere coloane .csv, url descriere), descriere sumara)}
# Descrierea sumara a fost dedusa din denumirile campurilor din fisierul .csv (descrierea coloanelor) al fiecarei perechi.
FINANCIARE_FILES = {
    2025: {
        "asiguratori": (
            ("WEBASIG2025.txt", "https://data.gov.ro/dataset/e2266fdc-0a6b-43b8-9c9f-7a6943d85b28/resource/fa2a915d-577d-405c-b7f2-753b0179abd4/download/webasig2025.txt"),
            ("WEBASIG2025.csv", "https://data.gov.ro/dataset/e2266fdc-0a6b-43b8-9c9f-7a6943d85b28/resource/205384e4-e4d9-4845-ac2c-3718380acc47/download/webasig2025.csv"),
            "Societati de asigurare-reasigurare: plasamente (investitii), rezerve tehnice, capitaluri proprii/fonduri mutuale si rezultatul tehnic pe asigurari generale si de viata.",
        ),
        "brokeri": (
            ("WEBBROK2025.txt", "https://data.gov.ro/dataset/e2266fdc-0a6b-43b8-9c9f-7a6943d85b28/resource/bc031b48-1275-4c83-87eb-3598031ec42c/download/webbrok2025.txt"),
            ("WEBBROK2025.csv", "https://data.gov.ro/dataset/e2266fdc-0a6b-43b8-9c9f-7a6943d85b28/resource/ee2c10dd-0aae-42b8-bfd4-2b58fc6a0073/download/webbrok2025.csv"),
            "Brokeri de asigurare-reasigurare: active imobilizate/circulante, capitaluri proprii, cifra de afaceri din activitatea de distributie si profitul curent.",
        ),
        "valori_sociale": (
            ("WEB_VS_2025.txt", "https://data.gov.ro/dataset/e2266fdc-0a6b-43b8-9c9f-7a6943d85b28/resource/6038ed97-0093-4040-9969-6638ed640c6f/download/web_vs_2025.txt"),
            ("WEB_VS_2025.csv", "https://data.gov.ro/dataset/e2266fdc-0a6b-43b8-9c9f-7a6943d85b28/resource/22f687c1-989b-421e-b2b9-4f9c16309528/download/web_vs_2025.csv"),
            "Bilant contabil sintetic (active imobilizate/circulante, capital, rezultat reportat, profit/pierdere) pentru entitatile de pe piata de capital raportate sub codul sursei 'VS'.",
        ),
        "sif": (
            ("WEB_SIF2025.txt", "https://data.gov.ro/dataset/e2266fdc-0a6b-43b8-9c9f-7a6943d85b28/resource/4f14f22e-c38b-4589-aa1d-3b612d997a9c/download/web_sif2025.txt"),
            ("WEB_SIF2025.csv", "https://data.gov.ro/dataset/e2266fdc-0a6b-43b8-9c9f-7a6943d85b28/resource/1cb20344-511e-4851-ba99-5291f47fbccd/download/web_sif2025.csv"),
            "Societati de Investitii Financiare (SIF): active imobilizate/circulante, capital, rezultat reportat si profitul/pierderea perioadei.",
        ),
        "pensii": (
            ("WEB_PENSII2025.txt", "https://data.gov.ro/dataset/e2266fdc-0a6b-43b8-9c9f-7a6943d85b28/resource/b120f7d4-306f-4239-9258-ba6d4732ad04/download/web_pensii2025.txt"),
            ("WEB_PENSII2025.csv", "https://data.gov.ro/dataset/e2266fdc-0a6b-43b8-9c9f-7a6943d85b28/resource/a0b1fad5-c870-4efc-99f6-54551521a800/download/web_pensii2025.csv"),
            "Administratori de fonduri de pensii private: active/capitaluri proprii, cifra de afaceri neta, venituri si cheltuieli totale, profit/pierdere.",
        ),
        "ifn": (
            ("WEB_IFN2025.txt", "https://data.gov.ro/dataset/e2266fdc-0a6b-43b8-9c9f-7a6943d85b28/resource/94317cfe-3f98-4041-9228-ec679196ba7f/download/web_ifn2025.txt"),
            ("WEB_IFN2025.csv", "https://data.gov.ro/dataset/e2266fdc-0a6b-43b8-9c9f-7a6943d85b28/resource/5fdd49f1-b304-4704-8f84-d67ebb6c3b94/download/web_ifn2025.csv"),
            "Institutii Financiare Nebancare (IFN): active si datorii financiare pe categorii IFRS 9 (detinute pentru tranzactionare, cost amortizat etc.), capital social si rezultatul exercitiului.",
        ),
        "ip_ieme": (
            ("WEB_IP_IEME2025.txt", "https://data.gov.ro/dataset/e2266fdc-0a6b-43b8-9c9f-7a6943d85b28/resource/4d9b9282-f3f2-4751-a014-72918911e0ae/download/web_ip_ieme2025.txt"),
            ("WEB_IP_IEME2025.csv", "https://data.gov.ro/dataset/e2266fdc-0a6b-43b8-9c9f-7a6943d85b28/resource/c182faf4-8dde-4d52-93db-a932399cd1b3/download/web_ip_ieme2025.csv"),
            "Institutii de Plata / Institutii Emitente de Moneda Electronica (IP/IEME): creante si datorii fata de institutii de credit si clientela, capital social si rezultatul activitatii curente.",
        ),
        "fond_garantare": (
            ("WEB_FOND_GARANTARE2025.txt", "https://data.gov.ro/dataset/e2266fdc-0a6b-43b8-9c9f-7a6943d85b28/resource/ad477e82-c502-42b1-a3b0-21b5e39e81ac/download/web_fond_garantare2025.txt"),
            ("WEB_FOND_GARANTARE2025.csv", "https://data.gov.ro/dataset/e2266fdc-0a6b-43b8-9c9f-7a6943d85b28/resource/693233d5-a4ea-4fc0-9ca8-45aca54df4c7/download/web_fond_garantare2025.csv"),
            "Fondul de garantare a depozitelor bancare: fondul aferent schemei de garantare, fondul de rezolutie bancara, provizioane si rezultatul activitatii curente.",
        ),
        "valori_mobiliare": (
            ("WEB_VM_AN2025.txt", "https://data.gov.ro/dataset/e2266fdc-0a6b-43b8-9c9f-7a6943d85b28/resource/c76323db-130c-4a67-a065-dacb744a6070/download/web_vm_an2025.txt"),
            ("WEB_VM_2025.csv", "https://data.gov.ro/dataset/e2266fdc-0a6b-43b8-9c9f-7a6943d85b28/resource/55ea40dd-7d63-4f24-ada5-e50148050aae/download/web_vm_2025.csv"),
            "Bilant contabil sintetic (structura similara cu SIF) pentru intermediari/societati de pe piata de capital raportate sub codul sursei 'VM'.",
        ),
        "institutii_de_credit": (
            ("WEB_INSTIT_DE_CREDIT_AN2025.txt", "https://data.gov.ro/dataset/e2266fdc-0a6b-43b8-9c9f-7a6943d85b28/resource/2322b065-8e43-4cbd-b825-2f45f3718c3b/download/web_instit_de_credit_an2025.txt"),
            ("WEB_Inst_de_credit_2025.csv", "https://data.gov.ro/dataset/e2266fdc-0a6b-43b8-9c9f-7a6943d85b28/resource/8e5a4a18-3737-4d4b-948c-19c4278981ba/download/web_inst_de_credit_2025.csv"),
            "Institutii de credit (banci): active si datorii financiare pe categorii IFRS 9, capital social, rezerve si rezultatul exercitiului.",
        ),
        "ong": (
            ("WEB_ONG_AN2025.txt", "https://data.gov.ro/dataset/e2266fdc-0a6b-43b8-9c9f-7a6943d85b28/resource/6df76210-1b50-42c5-a9cd-57d95c55631b/download/web_ong_an2025.txt"),
            ("WEB_ONG_AN2025.csv", "https://data.gov.ro/dataset/e2266fdc-0a6b-43b8-9c9f-7a6943d85b28/resource/44c8b0e0-0452-4a02-9af2-efadde59a467/download/web_ong_an2025.csv"),
            "ONG-uri (organizatii fara scop patrimonial): active/capitaluri, venituri si cheltuieli separate pe activitatea economica si pe activitatea fara scop patrimonial, cu excedentul aferent.",
        ),
        "ir": (
            ("WEB_IR_AN2025.txt", "https://data.gov.ro/dataset/e2266fdc-0a6b-43b8-9c9f-7a6943d85b28/resource/38a8cc80-3470-49a0-9335-7ae020d8239d/download/web_ir_an2025.txt"),
            ("WEB_IR_2025.csv", "https://data.gov.ro/dataset/e2266fdc-0a6b-43b8-9c9f-7a6943d85b28/resource/1f51b2ce-226e-41c0-9189-4192322067a2/download/web_ir_2025.csv"),
            "Bilant contabil sintetic ce include campul 'Patrimoniul regiei' - foarte probabil regii autonome (RA): active, capitaluri, cifra de afaceri, venituri/cheltuieli totale si profit/pierdere.",
        ),
        "uu": (
            ("WEB_UU_AN2025.txt", "https://data.gov.ro/dataset/e2266fdc-0a6b-43b8-9c9f-7a6943d85b28/resource/eeecc692-d914-4d3b-b7f5-d1a8a9791979/download/web_uu_an2025.txt"),
            ("WEB_UU_AN2025.csv", "https://data.gov.ro/dataset/e2266fdc-0a6b-43b8-9c9f-7a6943d85b28/resource/f5e400c8-3bfa-42fc-8d7b-01d389ac9e07/download/web_uu_an2025.csv"),
            "Bilant contabil sintetic, structura identica cu 'ir' (include tot 'Patrimoniul regiei') - raportat sub codul sursei 'UU', posibil alta categorie de unitati cu capital public.",
        ),
        "bl_bs_sl": (
            ("WEB_BL_BS_SL_AN2025.txt", "https://data.gov.ro/dataset/e2266fdc-0a6b-43b8-9c9f-7a6943d85b28/resource/3540a9ce-6a4d-4e29-9aa1-ac909fe28ac1/download/web_bl_bs_sl_an2025.txt"),
            ("WEB_BL_BS_SL_AN2025.csv", "https://data.gov.ro/dataset/e2266fdc-0a6b-43b8-9c9f-7a6943d85b28/resource/c4c57682-8764-4eb1-8431-0cac3ad6d7e5/download/web_bl_bs_sl_an2025.csv"),
            "Bilant contabil sintetic generic (active, capitaluri, cifra de afaceri, venituri/cheltuieli totale, profit/pierdere); codul sursei 'BL_BS_SL' sugereaza gruparea celor 3 tipuri de bilant ANAF (lung/scurt/simplificat).",
        ),
    },
    # 2024: {...}   # de completat cand descarcam si anul 2024 (fisiere + URL-uri noi de pe data.gov.ro)
}


def resolve_file(filename: str, url: str, dest_dir: Path) -> Path:
    path = dest_dir / filename
    if not path.exists():
        if not DOWNLOAD_MISSING_FILES:
            raise FileNotFoundError(
                f"{path} nu exista. Seteaza DOWNLOAD_MISSING_FILES = True ca sa fie descarcat automat."
            )
        dest_dir.mkdir(parents=True, exist_ok=True)
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        path.write_bytes(response.content)
    return path


def slugify_descriere(text: str, max_len: int = 80) -> str:
    """Transforma o descriere in limba romana intr-un nume de coloana simplu (ascii, snake_case)."""
    text = unicodedata.normalize("NFKD", str(text))
    text = text.encode("ascii", "ignore").decode("ascii")
    text = text.lower()
    text = re.sub(r"[^a-z0-9]+", "_", text).strip("_")
    text = re.sub(r"_+", "_", text)
    return text[:max_len].rstrip("_")


def construieste_mapare_coloane(df_columns: pd.DataFrame) -> dict:
    """Cod (ex. I12) -> nume de coloana sugestiv, dedus din descriere. Dezambiguizeaza denumirile identice."""
    mapare = {}
    denumiri_folosite = {}
    for _, rand in df_columns.iterrows():
        cod = str(rand["cod"]).strip().upper()
        if cod.isdigit():
            # cateva coduri din fisierele sursa lipsesc prefixul "I" (eroare in datele publicate de sursa)
            cod = f"I{cod}"
        denumire = slugify_descriere(rand["descriere"]) or cod.lower()
        n = denumiri_folosite.get(denumire, 0)
        denumiri_folosite[denumire] = n + 1
        if n > 0:
            denumire = f"{denumire}_{n + 1}"
        mapare[cod] = denumire
    return mapare


financiare = {}

if is_active("data_gov_ro"):
    for an in ANI_ACTIVI:
        dir_an = DATA_GOV_RO_BASE_DIR / f"Finance_{an}"
        financiare[an] = {}

        for key, ((data_file, data_url), (columns_file, columns_url), descriere) in FINANCIARE_FILES[an].items():
            data_path = resolve_file(data_file, data_url, dir_an)
            columns_path = resolve_file(columns_file, columns_url, dir_an)

            df_data = pd.read_csv(data_path, encoding="utf-8")
            # cp1250 (nu cp1252/latin1!): fisierele sursa folosesc codificarea Windows Europa Centrala,
            # necesara pentru diacriticele romanesti (ă/â/î/ș/ț) din descrierile coloanelor.
            df_columns = pd.read_csv(
                columns_path, sep=";", header=None, names=["descriere", "cod"], encoding="cp1250"
            )

            mapare_coloane = construieste_mapare_coloane(df_columns)
            coloane_nemapate = set(df_data.columns) - set(mapare_coloane.keys())
            if coloane_nemapate:
                print(f"[{an}/{key}] atentie: coloane fara descriere gasita, raman cu numele original: {coloane_nemapate}")

            df_data_denumit = df_data.rename(columns=mapare_coloane)

            financiare[an][key] = {
                "data": df_data,                    # coloane originale (CUI, CAEN, I1, I2, ...) - stabile, conform sursei
                "data_denumit": df_data_denumit,     # aceleasi date, coloane redenumite dupa descriere
                "columns": df_columns,
                "mapare_coloane": mapare_coloane,    # cod -> nume sugestiv, pentru trasabilitate inversa
                "descriere": descriere,
            }

    rezumat_financiare = pd.DataFrame(
        [
            {
                "an": an,
                "sursa": key,
                "descriere": dfs["descriere"],
                "randuri": dfs["data"].shape[0],
                "coloane_date": dfs["data"].shape[1],
                "coloane_descrise": dfs["columns"].shape[0],
            }
            for an, seturi in financiare.items()
            for key, dfs in seturi.items()
        ]
    )
    display(rezumat_financiare)

    # Un esantion (primele 3 randuri) din fiecare tabel, cu coloanele redenumite sugestiv.
    # Codurile originale (I1, I2, ...) raman disponibile in financiare[an][cheie]["data"]
    # si maparea lor completa in financiare[an][cheie]["mapare_coloane"].
    for an, seturi in financiare.items():
        for key, dfs in seturi.items():
            display(HTML(f"<h4>{an} / {key}</h4><p>{dfs['descriere']}</p>"))
            display(dfs["data_denumit"].head(3))
else:
    rezumat_financiare = pd.DataFrame()
    print("data_gov_ro: sarit (dezactivat in ACTIVE_SOURCES)")

## Sursa: Guvernul Romaniei - Firme inregistrate la Registrul Comertului (data.gov.ro)

Dataset: https://data.gov.ro/dataset/firme-03-02-2026

6 fisiere `.csv`, toate delimitate prin `^` (nu prin `,` sau `;`), encoding UTF-8 cu BOM, cu header in clar (nu mai exista fisier separat de descriere a coloanelor ca la situatiile financiare).

- **`OD_FIRME`** - tabelul principal: `DENUMIRE`, `CUI`, `COD_INMATRICULARE`, `DATA_INMATRICULARE`, `FORMA_JURIDICA`, adresa (`ADR_*`), `WEB`, `TARA_FIRMA_MAMA`. **Nu contine un cod CAEN** — codurile CAEN (activitatea) sunt doar in `OD_CAEN_AUTORIZAT`, legate prin `COD_INMATRICULARE`.
- **`OD_CAEN_AUTORIZAT`** - codurile CAEN autorizate per firma (relatie 1:N fata de `OD_FIRME`), cu versiunea nomenclatorului CAEN (`VER_CAEN_AUTORIZAT`).
- **`OD_STARE_FIRMA`** - un cod numeric de stare per `COD_INMATRICULARE` (119 coduri distincte in date). **Nu exista o legenda a codurilor in acest dataset** — nu stim ce inseamna fiecare cod fara o sursa externa.
- **`OD_REPREZENTANTI_LEGALI`**, **`OD_REPREZENTANTI_IF`** - reprezentanti legali / ai intreprinderilor familiale per firma.
- **`OD_SUCURSALE_ALTE_STATE_MEMBRE`** - sucursale infiintate in alte state membre UE (fisier mic).

Cheia de legatura intre fisiere este `COD_INMATRICULARE` (numarul de ordine in registru, ex. `J40/1116/1991`), nu `CUI` — `CUI` lipseste pentru o mica parte din firme (14 randuri din ~4.1M au `CUI` gol in sursa).

**Atentie la memorie**: `OD_FIRME` (~678MB, ~4.14M randuri) si `OD_CAEN_AUTORIZAT` (~406MB, ~18.5M randuri) ocupa impreuna cateva GB in RAM odata incarcate ca DataFrame-uri; codul de mai jos foloseste dtype `category` pentru coloanele cu putine valori distincte, ca sa reduca amprenta in memorie.

Implicit (`DOWNLOAD_MISSING_FILES_FIRME = False`), celula de mai jos doar incarca fisierele deja existente in `data.gov.ro/firme-03-02-2026/` — nu face niciun apel de retea.

In [ ]:
FIRME_DIR = Path("/Users/tudor/Documents/Data-for-Projects/Cercetare-Research/data.gov.ro/firme-03-02-2026")

# Daca False (implicit), nu se face niciun apel de retea: se incarca doar fisierele deja
# prezente in FIRME_DIR. Seteaza pe True doar cand vrei sa (re)descarci fisierele lipsa.
DOWNLOAD_MISSING_FILES_FIRME = False

FIRME_BASE_URL = "https://data.gov.ro/dataset/02ee9ace-19cc-4ca3-b142-9aa9aa968ab3/resource"

# Fiecare intrare: fisier, url, descriere, dtype-uri optionale (category pt. coloanele cu putine valori distincte)
FIRME_FILES = {
    "firme": (
        "OD_FIRME.csv",
        f"{FIRME_BASE_URL}/dbe0623a-ff87-460a-bafe-69f1564bbb4b/download/od_firme.csv",
        "Tabelul principal: denumire, CUI, cod inmatriculare, data inmatricularii, forma juridica si adresa. Nu contine CAEN.",
        {"CUI": "string", "FORMA_JURIDICA": "category", "ADR_TARA": "category",
         "ADR_JUDET": "category", "ADR_SECTOR": "category", "TARA_FIRMA_MAMA": "category"},
    ),
    "caen_autorizat": (
        "OD_CAEN_AUTORIZAT.csv",
        f"{FIRME_BASE_URL}/9260a779-dbd1-4222-a0b8-3e8039ae455d/download/od_caen_autorizat.csv",
        "Codurile CAEN autorizate per firma (1:N fata de firme), cu versiunea nomenclatorului CAEN.",
        {"COD_CAEN_AUTORIZAT": "category", "VER_CAEN_AUTORIZAT": "category"},
    ),
    "stare_firma": (
        "OD_STARE_FIRMA.csv",
        f"{FIRME_BASE_URL}/7259bbf8-7206-4990-83d2-f55ad00d5013/download/od_stare_firma.csv",
        "Cod numeric de stare per firma. Fara legenda publicata in acest dataset - codurile raman needescifrate.",
        {"COD": "category"},
    ),
    "reprezentanti_legali": (
        "OD_REPREZENTANTI_LEGALI.csv",
        f"{FIRME_BASE_URL}/dfb3e9fa-ea59-49e8-802c-83fb83b46052/download/od_reprezentanti_legali.csv",
        "Reprezentanti legali / persoane imputernicite per firma (administratori, lichidatori etc.).",
        {},
    ),
    "reprezentanti_if": (
        "OD_REPREZENTANTI_IF.csv",
        f"{FIRME_BASE_URL}/8e5b1305-37da-4236-aff4-28ba419236b5/download/od_reprezentanti_if.csv",
        "Reprezentanti / membri ai intreprinderilor familiale (IF).",
        {},
    ),
    "sucursale_alte_state_membre": (
        "OD_SUCURSALE_ALTE_STATE_MEMBRE.csv",
        f"{FIRME_BASE_URL}/2136f7b4-5472-4b81-9cb5-39b2a26c2809/download/od_sucursale_alte_state_membre.csv",
        "Sucursale ale firmelor romanesti infiintate in alte state membre UE.",
        {},
    ),
}


def resolve_file_firme(filename: str, url: str, dest_dir: Path = FIRME_DIR) -> Path:
    path = dest_dir / filename
    if not path.exists():
        if not DOWNLOAD_MISSING_FILES_FIRME:
            raise FileNotFoundError(
                f"{path} nu exista. Seteaza DOWNLOAD_MISSING_FILES_FIRME = True ca sa fie descarcat automat."
            )
        dest_dir.mkdir(parents=True, exist_ok=True)
        response = requests.get(url, timeout=120)
        response.raise_for_status()
        path.write_bytes(response.content)
    return path


firme_2026 = {}

if is_active("firme_data_gov_ro"):
    for key, (filename, url, descriere, dtypes) in FIRME_FILES.items():
        path = resolve_file_firme(filename, url)
        df = pd.read_csv(path, sep="^", encoding="utf-8-sig", dtype=dtypes or None, low_memory=False)
        firme_2026[key] = {"data": df, "descriere": descriere}

    rezumat_firme_2026 = pd.DataFrame(
        [
            {
                "sursa": key,
                "descriere": dfs["descriere"],
                "randuri": dfs["data"].shape[0],
                "coloane": dfs["data"].shape[1],
            }
            for key, dfs in firme_2026.items()
        ]
    )
    display(rezumat_firme_2026)

    for key, dfs in firme_2026.items():
        display(HTML(f"<h4>{key}</h4><p>{dfs['descriere']}</p>"))
        display(dfs["data"].head(3))
else:
    rezumat_firme_2026 = pd.DataFrame()
    print("firme_data_gov_ro: sarit (dezactivat in ACTIVE_SOURCES)")

## Surse noi

Pe masura ce descoperim surse noi de date, le adaugam aici: o cheie noua in `ACTIVE_SOURCES` mai sus, plus o pereche de celule (markdown + cod) similara cu cele de deasupra.